In [0]:
import numpy as np
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# --- Configuración inicial de variables de prueba ---
# (Se inicializan solo si no las has definido previamente en tu notebook)
N = 100 if 'N' not in globals() else globals()['N']
restaurantes = ["Bembos", "KFC", "Pardos Chicken", "La Lucha", "Siete Sopas"] if 'restaurantes' not in globals() else globals()['restaurantes']
distritos = ["Miraflores", "San Isidro", "Surco", "Lince", "San Borja"] if 'distritos' not in globals() else globals()['distritos']
estados = ["ENTREGADO", "EN_CAMINO", "CANCELADO", "PREPARANDO", "PROGRAMADO"] if 'estados' not in globals() else globals()['estados']

# 1. Definimos el ESQUEMA EXPLÍCITO para evitar errores de inferencia
schema = StructType([
    StructField("id_pedido", StringType(), True),
    StructField("restaurante", StringType(), True),
    StructField("distrito", StringType(), True),
    StructField("hora", IntegerType(), True),
    StructField("monto_soles", DoubleType(), True),
    StructField("costo_delivery", DoubleType(), True),
    StructField("tiempo_entrega_min", IntegerType(), True),
    StructField("rating", DoubleType(), True),
    StructField("id_repartidor", StringType(), True),
    StructField("estado", StringType(), True)
])

# 2. Generamos la lista de tuplas con casteos explícitos a tipos nativos de Python (str, int, float)
datos_pedidos = [
    (
        f"PD{i:06d}",
        str(np.random.choice(restaurantes)),          # <-- CORREGIDO: Convertido a str estándar de Python
        str(np.random.choice(distritos)),             # <-- CORREGIDO: Convertido a str estándar de Python
        int(np.random.randint(7, 23)),                # Convertido a int estándar
        float(round(np.random.uniform(18, 120), 2)),  # Convertido a float estándar
        float(round(np.random.uniform(3, 12), 2)),    # Convertido a float estándar
        int(np.random.randint(15, 65)),               # Convertido a int estándar
        float(round(np.random.normal(4.2, 0.5), 1)),  # Convertido a float estándar
        f"REP{np.random.randint(1, 40):03d}",
        str(np.random.choice(estados, p=[.75, .08, .05, .10, .02]))
    )
    for i in range(1, N + 1)
]

# 3. Creamos el DataFrame pasando los datos y el esquema definido
pedidos = spark.createDataFrame(datos_pedidos, schema=schema)

# 4. Visualizamos los datos inmediatamente en Databricks
display(pedidos)

id_pedido,restaurante,distrito,hora,monto_soles,costo_delivery,tiempo_entrega_min,rating,id_repartidor,estado
PD000001,Astrid & Gastón,Barranco,21,111.6,9.06,28,4.0,REP030,entregado
PD000002,Bembos,Callao,8,103.94,7.51,20,4.1,REP029,cancelado
PD000003,Tanta,Callao,21,24.69,7.94,48,4.9,REP006,entregado
PD000004,Sushi Pop,Barranco,22,95.74,9.42,47,5.0,REP032,entregado
PD000005,Sushi Pop,Callao,16,64.13,4.29,49,4.7,REP024,entregado
PD000006,KFC Lima,SJL,13,67.5,10.0,52,3.6,REP008,entregado
PD000007,Sushi Pop,San Isidro,22,79.31,10.64,47,5.2,REP016,entregado
PD000008,La Mar,Miraflores,19,75.32,5.83,19,4.3,REP010,entregado
PD000009,Astrid & Gastón,Callao,19,73.67,6.23,60,5.2,REP003,entregado
PD000010,Pizza Hut Miraflores,Callao,7,21.83,9.45,50,3.9,REP039,entregado


In [0]:
# ============================================================
# CELDA 2: Consulta 1 — Top 5 restaurantes por ingresos
# ============================================================

# Registrar el DataFrame como vista temporal para usar SQL
pedidos.createOrReplaceTempView("pedidos")

# Consulta SQL
top_restaurantes = spark.sql("""
SELECT
    restaurante,
    COUNT(*) AS total_pedidos,
    ROUND(SUM(monto_soles), 2) AS ingresos_totales,
    ROUND(AVG(rating), 2) AS rating_prom,
    ROUND(AVG(tiempo_entrega_min), 1) AS tiempo_entrega_prom
FROM pedidos
WHERE estado = 'entregado'
GROUP BY restaurante
ORDER BY ingresos_totales DESC
LIMIT 5
""")

# Mostrar resultados
print("🍽️ TOP 5 RESTAURANTES — INGRESOS DEL MES")
display(top_restaurantes)

🍽️ TOP 5 RESTAURANTES — INGRESOS DEL MES


restaurante,total_pedidos,ingresos_totales,rating_prom,tiempo_entrega_prom
Sushi Pop,81,5557.28,4.28,39.5
Pardos Chicken,78,5390.22,4.17,39.2
Cevichería Isolina,74,5389.82,4.27,39.7
Pizza Hut Miraflores,78,5360.86,4.21,41.4
El Hornero,70,5237.97,4.3,38.8


In [0]:
# ============================================================
# CELDA 3: Consulta 2 — Demanda por hora
# ¿A qué hora del día hay más pedidos y mayor ingreso?
# ============================================================

# Asegurar que la vista temporal exista
pedidos.createOrReplaceTempView("pedidos")

demanda_hora = spark.sql("""
SELECT
    hora,
    COUNT(*) AS total_pedidos,
    ROUND(SUM(monto_soles), 2) AS ingresos_hora,
    ROUND(AVG(tiempo_entrega_min), 1) AS tiempo_prom
FROM pedidos
WHERE estado = 'entregado'
GROUP BY hora
ORDER BY total_pedidos DESC
""")

print("🕐 DEMANDA POR HORA DEL DÍA")

display(demanda_hora)

🕐 DEMANDA POR HORA DEL DÍA


hora,total_pedidos,ingresos_hora,tiempo_prom
13,53,3355.73,39.8
11,51,3838.84,41.7
18,51,3160.32,38.8
19,50,3686.67,39.2
14,50,3721.36,40.1
21,48,3478.69,39.2
16,47,3210.62,37.8
9,47,3308.29,38.1
12,47,3294.37,44.6
7,44,2866.88,39.0


In [0]:
# ============================================================
# CELDA 4: Consulta 3 — Repartidores eficientes
# - Más de 10 pedidos entregados
# - Tiempo promedio < 40 min
# - Rating promedio > 4.0
# ============================================================

# Asegurar que la vista temporal exista
pedidos.createOrReplaceTempView("pedidos")

repartidores_eficientes = spark.sql("""
SELECT
    id_repartidor,
    COUNT(*) AS pedidos_entregados,
    ROUND(AVG(tiempo_entrega_min), 1) AS tiempo_prom,
    ROUND(AVG(rating), 2) AS rating_prom,
    ROUND(SUM(costo_delivery), 2) AS ingresos_delivery
FROM pedidos
WHERE estado = 'entregado'
GROUP BY id_repartidor
HAVING COUNT(*) > 10
   AND AVG(tiempo_entrega_min) < 40
   AND AVG(rating) > 4.0
ORDER BY rating_prom DESC
""")

print("⚡ REPARTIDORES EFICIENTES (>10 pedidos, tiempo < 40 min, rating > 4.0)")

display(repartidores_eficientes)

print(f"Total repartidores que cumplen los criterios: {repartidores_eficientes.count()}")

⚡ REPARTIDORES EFICIENTES (>10 pedidos, tiempo < 40 min, rating > 4.0)


id_repartidor,pedidos_entregados,tiempo_prom,rating_prom,ingresos_delivery
REP039,15,37.8,4.53,124.86
REP017,15,36.9,4.44,127.81
REP008,29,39.6,4.4,210.32
REP010,21,39.6,4.39,154.88
REP011,16,30.3,4.38,121.35
REP005,22,36.9,4.36,179.86
REP025,21,37.4,4.32,174.23
REP016,23,39.3,4.31,154.91
REP015,17,37.7,4.31,128.18
REP027,18,38.0,4.25,131.31


Total repartidores que cumplen los criterios: 21


In [0]:
# ============================================================
# CELDA 5: Respuestas
# ============================================================

pregunta_1 = """
¿Cuándo hay más pedidos según tu consulta (hora del día)?
¿Coincide con lo que esperabas para Lima? ¿Por qué sí o no?

# Tu respuesta:
Los mayores pedidos se concentran en las horas de la noche, aproximadamente entre las 19:00 y 21:00 horas.
Esto coincide con el comportamiento esperado en Lima, ya que en ese rango horario la mayoría de personas ya ha salido de trabajar o estudiar y realiza pedidos de comida a domicilio.
"""

pregunta_2 = """
Si la empresa quiere bonificar a los mejores repartidores,
¿cuál de los 3 criterios de la Consulta 3 es el más importante?
Justifica con un argumento de negocio.

# Tu respuesta:
El criterio más importante es el tiempo promedio de entrega, porque impacta directamente en la experiencia del cliente.
Un reparto rápido mejora la satisfacción, aumenta la probabilidad de recompra y reduce cancelaciones, lo cual tiene un impacto directo en los ingresos del negocio.
"""

pregunta_3 = """
¿Por qué usamos Spark SQL y no pandas para este análisis?
Da UN argumento técnico específico.

# Tu respuesta:
Usamos Spark SQL porque permite procesar grandes volúmenes de datos de forma distribuida, lo que mejora el rendimiento y escalabilidad frente a pandas, que trabaja en memoria en un solo nodo.
"""

print("✅ Respuestas registradas")
print(pregunta_1)
print(pregunta_2)
print(pregunta_3)

✅ Respuestas registradas

¿Cuándo hay más pedidos según tu consulta (hora del día)?
¿Coincide con lo que esperabas para Lima? ¿Por qué sí o no?

# Tu respuesta:
Los mayores pedidos se concentran en las horas de la noche, aproximadamente entre las 19:00 y 21:00 horas.
Esto coincide con el comportamiento esperado en Lima, ya que en ese rango horario la mayoría de personas ya ha salido de trabajar o estudiar y realiza pedidos de comida a domicilio.


Si la empresa quiere bonificar a los mejores repartidores,
¿cuál de los 3 criterios de la Consulta 3 es el más importante?
Justifica con un argumento de negocio.

# Tu respuesta:
El criterio más importante es el tiempo promedio de entrega, porque impacta directamente en la experiencia del cliente.
Un reparto rápido mejora la satisfacción, aumenta la probabilidad de recompra y reduce cancelaciones, lo cual tiene un impacto directo en los ingresos del negocio.


¿Por qué usamos Spark SQL y no pandas para este análisis?
Da UN argumento técnico e